# Machine Learning No Supervisado: Clustering y Reducción de Dimensionalidad
Informe práctico — K-means, Clustering Jerárquico, PCA y SVD con Scikit-learn.

**Autor:** Jerson Baquero Cháves · Especialización en Inteligencia Artificial, Universidad de Cundinamarca

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram

os.makedirs('figs', exist_ok=True)
RS = 42

## 1. Datos: Wine (13 variables químicas, 178 muestras)
Se ignoran las etiquetas durante el entrenamiento y solo se usan al final para validar los grupos (ARI).

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
print(X.shape)
Xs = StandardScaler().fit_transform(X)   # escalado: K-means y PCA son sensibles a la escala

## 2. K-means: elección de k (método del codo y silueta)

In [ ]:
ks = range(2, 9)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RS).fit(Xs)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xs, km.labels_))
    print(f'k={k}  inercia={km.inertia_:.1f}  silueta={sils[-1]:.3f}')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(list(ks), inertias, 'o-'); ax[0].set_title('Método del codo'); ax[0].set_xlabel('k'); ax[0].set_ylabel('Inercia')
ax[1].plot(list(ks), sils, 'o-', color='tab:green'); ax[1].set_title('Coeficiente de silueta'); ax[1].set_xlabel('k')
plt.tight_layout(); plt.savefig('figs/kmeans_codo_silueta.png', dpi=150); plt.show()

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=RS).fit(Xs)
print('Silueta k=3:', round(silhouette_score(Xs, km.labels_), 3))
print('ARI vs. etiquetas reales:', round(adjusted_rand_score(y, km.labels_), 3))
print(pd.crosstab(y, km.labels_, rownames=['real'], colnames=['cluster']))

## 3. Clustering jerárquico aglomerativo (Ward)

In [ ]:
Z = linkage(Xs, method='ward')
plt.figure(figsize=(9, 4))
dendrogram(Z, truncate_mode='lastp', p=20, leaf_rotation=90)
plt.title('Dendrograma (Ward, últimos 20 grupos)'); plt.xlabel('Muestras / grupos'); plt.ylabel('Distancia')
plt.tight_layout(); plt.savefig('figs/dendrograma.png', dpi=150); plt.show()

hc = AgglomerativeClustering(n_clusters=3, linkage='ward').fit(Xs)
print('Silueta jerárquico:', round(silhouette_score(Xs, hc.labels_), 3))
print('ARI vs. etiquetas reales:', round(adjusted_rand_score(y, hc.labels_), 3))
print('ARI K-means vs. jerárquico:', round(adjusted_rand_score(km.labels_, hc.labels_), 3))

## 4. PCA sobre Wine

In [ ]:
pca = PCA().fit(Xs)
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)
print('Varianza explicada (primeros 5 PC):', np.round(evr[:5], 3))
print('Componentes para >=90% de varianza:', int(np.argmax(cum >= 0.90)) + 1)

Z2 = PCA(n_components=2).fit_transform(Xs)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(range(1, len(cum)+1), cum, 'o-'); ax[0].axhline(0.9, ls='--', c='gray')
ax[0].set_title('Varianza acumulada (PCA)'); ax[0].set_xlabel('Componentes')
ax[1].scatter(Z2[:, 0], Z2[:, 1], c=km.labels_, cmap='viridis', s=25); ax[1].set_title('K-means (k=3) en el plano PCA')
ax[2].scatter(Z2[:, 0], Z2[:, 1], c=y, cmap='viridis', s=25); ax[2].set_title('Clases reales en el plano PCA')
for a in ax[1:]: a.set_xlabel('PC1'); a.set_ylabel('PC2')
plt.tight_layout(); plt.savefig('figs/pca_wine.png', dpi=150); plt.show()

# K-means sobre los datos reducidos a 2 componentes
km2 = KMeans(n_clusters=3, n_init=10, random_state=RS).fit(Z2)
print('ARI K-means en 2 PC:', round(adjusted_rand_score(y, km2.labels_), 3))

# Variables con mayor peso en PC1 y PC2
loadings = pd.DataFrame(pca.components_[:2].T, index=X.columns, columns=['PC1', 'PC2'])
print(loadings.reindex(loadings.PC1.abs().sort_values(ascending=False).index).head(5).round(3))

## 5. SVD truncada (TruncatedSVD) sobre Digits (64 variables, imágenes 8x8)

In [ ]:
digits = load_digits()
Xd, yd = digits.data, digits.target
print(Xd.shape)

svd = TruncatedSVD(n_components=40, random_state=RS).fit(Xd)
cum_svd = np.cumsum(svd.explained_variance_ratio_)
pca_d = PCA(n_components=40, random_state=RS).fit(Xd)
cum_pca = np.cumsum(pca_d.explained_variance_ratio_)
print('Varianza con 10 comp. -> SVD: %.3f | PCA: %.3f' % (cum_svd[9], cum_pca[9]))
print('Varianza con 40 comp. -> SVD: %.3f | PCA: %.3f' % (cum_svd[-1], cum_pca[-1]))

Zs = TruncatedSVD(n_components=2, random_state=RS).fit_transform(Xd)
Zp = PCA(n_components=2).fit_transform(Xd)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(range(1, 41), cum_svd, label='SVD'); ax[0].plot(range(1, 41), cum_pca, '--', label='PCA')
ax[0].set_title('Varianza acumulada'); ax[0].set_xlabel('Componentes'); ax[0].legend()
for a, Zm, t in zip(ax[1:], [Zs, Zp], ['SVD truncada (2D)', 'PCA (2D)']):
    sc = a.scatter(Zm[:, 0], Zm[:, 1], c=yd, cmap='tab10', s=8); a.set_title(t)
plt.colorbar(sc, ax=ax[2], label='Dígito')
plt.tight_layout(); plt.savefig('figs/svd_pca_digits.png', dpi=150); plt.show()

In [ ]:
# Clustering sobre Digits: datos originales vs. reducidos con SVD (20 componentes)
Xr = TruncatedSVD(n_components=20, random_state=RS).fit_transform(Xd)
for nombre, M in [('Original (64 var.)', Xd), ('SVD (20 comp.)', Xr)]:
    k10 = KMeans(n_clusters=10, n_init=10, random_state=RS).fit(M)
    print(f'{nombre}: silueta={silhouette_score(M, k10.labels_):.3f}  ARI={adjusted_rand_score(yd, k10.labels_):.3f}')

# Reconstrucción de un dígito con distinto número de componentes
fig, ax = plt.subplots(1, 4, figsize=(9, 2.6))
ax[0].imshow(Xd[0].reshape(8, 8), cmap='gray_r'); ax[0].set_title('Original')
for a, n in zip(ax[1:], [5, 15, 30]):
    p = PCA(n_components=n).fit(Xd)
    rec = p.inverse_transform(p.transform(Xd[:1]))
    a.imshow(rec.reshape(8, 8), cmap='gray_r'); a.set_title(f'{n} comp.')
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig('figs/reconstruccion.png', dpi=150); plt.show()